# 🟢 cash — live feature tour

[cash](https://github.com/galgtonold/cash) caches your notebook **one statement at a time** and tracks how cells depend on each other, so re-running only recomputes what actually changed — nothing else.

This tour builds a small transaction-analytics pipeline and shows what cash does for you along the way. The whole thing runs in about a minute.

**How to use this notebook**
1. **Run all.** Every cell runs once and prints a cash **badge** summarizing what it did (`EXECUTED`, in ochre).
2. **Run all a _second_ time.** The expensive cells flip to green **`CACHED`** — served instantly from cache. Nothing recomputes.
3. **Edit `TOP_N`** (§3) and Run all — only that one leaf cell recomputes.
4. **Edit an upstream setting** `MIN_SPEND` (§4) and run *just the summary cell below it* — cash re-runs the steps in between for you and leaves the 2-million-row pipeline cached.
5. **Edit the helper** `spending_tier` (§5) — every cached result that calls it, directly or through another function, invalidates on its own.
6. **Restart the kernel** and Run all — the cache survives, so the slow steps still restore.

> ### ⚠️ Save the notebook before you run
>
> Steps 4 and 5 edit one cell and then run a **different** one. cash reads the cells it *didn't* execute from the **saved `.ipynb` file** — so an edit you haven't saved is invisible to it. It reads the old value, concludes nothing upstream changed, and hands you the previous answer while your screen shows the new code.
>
> **In JupyterLab (this Binder) and VS Code: press `Ctrl+S` / `Cmd+S` after editing, then run.** Autosave exists, but it runs on a timer that is usually slower than you are. On Google Colab there is nothing to save — cash reads the cells live from the frontend.

In [ ]:
# Install cash from PyPI. The [pandas] extra pulls pandas + pyarrow; numpy,
# pandas and matplotlib are already present on Colab. (On a re-run this is a
# fast no-op — pip sees it's already satisfied.)
%pip install -q "cash-lib[pandas]"

import cash
import numpy as np
import pandas as pd

%cash_on

## 1 · An expensive pipeline that caches itself

First, a million synthetic transactions to work with. It's **seeded**, so the
data is identical every run. Building it is quick — the raw data is not the
expensive part of this pipeline, and it isn't what makes caching worth having.

In [ ]:
rng = np.random.default_rng(0)
N = 1_000_000

transactions = pd.DataFrame({
    "user_id":  rng.integers(1, 500_000, N),
    "category": rng.choice(["groceries", "travel", "tech", "health", "dining"], N),
    "amount":   rng.gamma(shape=2.0, scale=40.0, size=N).round(2),
})

print(f"{len(transactions):,} transactions  |  ${transactions.amount.sum():,.0f} total spend")
transactions.head()

Then a grouped aggregation with a distinct-count (`nunique`). Also quick. Both
of these are ordinary pandas work that cash will hand back unchanged next run.

In [ ]:
category_stats = (
    transactions
    .groupby("category")
    .agg(total_spend=("amount", "sum"),
         avg_spend=("amount", "mean"),
         txns=("amount", "size"),
         unique_users=("user_id", "nunique"))
    .sort_values("total_spend", ascending=False)
)
category_stats

Now the expensive part — and notice it is a *computation*, not a pile of data.
A bootstrap resamples each category a few hundred times to put a confidence
interval around its mean transaction. That takes real seconds of CPU, it is
**seeded** so the answer is identical every run, and the result is five rows.

That last detail is the one worth internalising: **cash stores the answer, not
the work.** A slow computation with a small result is almost free to cache —
the entry here is about a kilobyte. A fast computation with a huge result is
the opposite, and the badge's `OVERHEAD` row will tell you when you have one.

In [ ]:
# Bootstrap a 95% CI for the mean transaction in each category.
# Seconds of genuine compute — and the result is a five-row table.
BOOTSTRAP_ITERS = 200

sim_rng = np.random.default_rng(42)
rows = {}

for category, group in transactions.groupby("category", observed=True):
    amounts = group["amount"].to_numpy()
    means = np.empty(BOOTSTRAP_ITERS)
    for i in range(BOOTSTRAP_ITERS):
        means[i] = sim_rng.choice(amounts, size=len(amounts), replace=True).mean()
    lo, hi = np.percentile(means, [2.5, 97.5])
    rows[category] = (amounts.mean(), lo, hi)

category_risk = pd.DataFrame(rows, index=["mean", "ci_lo", "ci_hi"]).T.round(2)
category_risk

## 2 · Run it again — and it's free

Hit **Run all** a second time now. Watch the cells above: their badges flip
from ochre `EXECUTED` to green **`CACHED`**, and they finish instantly. cash
recognised that neither the code nor its inputs changed, so it handed back the
cached values instead of recomputing them.

## 3 · Smart invalidation — change one thing, recompute one thing

The cell below depends on `category_stats` (cached above) but is cheap.
**Change `TOP_N` to another number and Run all again:** only this cell
recomputes — the expensive simulation above stays cached and instant.

In [ ]:
# 👇  EDIT THIS NUMBER and Run all again — only this cell recomputes.
TOP_N = 3

top = category_stats.head(TOP_N)

import matplotlib.pyplot as plt
ax = top["total_spend"][::-1].plot(kind="barh", color="#2e9e6b")
ax.set_title(f"Top {TOP_N} categories by total spend")
ax.set_xlabel("total spend ($)")
plt.tight_layout()
plt.show()

top

## 4 · Change an _upstream_ setting — re-run just the tail

This is the part that makes cash feel like magic. `MIN_SPEND` below is a config
value the next step depends on. Change it, **save the notebook** (`Ctrl+S` /
`Cmd+S`), then run **only the summary cell at the end of this section** (select
it and press Ctrl/Cmd+Enter — don't run the cells in between).

cash sees that the filtered aggregation is now stale, re-runs *that* step for
you, and leaves the `transactions` frame cached. You edit one
setting and ask for the answer; cash works out the minimum it has to redo.

> **If the number doesn't change, you skipped the save.** cash reads the cell you
> edited but didn't run from the file on disk, so an unsaved `MIN_SPEND` is still
> the old `MIN_SPEND` as far as it can tell. (Not applicable on Colab, where cash
> reads cells live.)

In [ ]:
# ⚙️  Upstream setting. Change it, then run just the summary cell below —
#     not this one, and not the frame above.
MIN_SPEND = 50

In [ ]:
# Total spend per user, counting only transactions of at least MIN_SPEND.
# Filtering + grouping a million rows takes a moment — and it depends on
# MIN_SPEND, so cash re-runs it whenever that setting changes.
qualified      = transactions[transactions["amount"] >= MIN_SPEND]
spend_per_user = qualified.groupby("user_id")["amount"].sum()

spend_per_user.describe().round(2)

In [ ]:
# 👇  Run ONLY this cell after changing MIN_SPEND. cash re-runs the filter and
#     grouping above (they depend on MIN_SPEND) but restores `transactions`.
whales = (spend_per_user > 2_000).sum()
print(f"{whales:,} users spent over $2,000 total   (min per-transaction: ${MIN_SPEND})")

## 5 · Edit a helper — everything that calls it updates

cash hashes a function's source **and** the source of the functions it calls.
So when you change an inner helper, every cached result that reaches it —
directly or transitively — invalidates on its own.

Edit a threshold in `spending_tier` below, **save** (`Ctrl+S` / `Cmd+S`), and
re-run the summary cell under it. You never touch `summarize_tiers`, yet its
cached answer updates, because it calls the helper you changed.

> Same rule as §4: the helper cell is one you edit but don't run, so cash only
> sees the edit once it's on disk.

In [ ]:
# A small helper. Edit a threshold, then re-run the summary cell below.
def spending_tier(total):
    if total > 2_000: return "whale"
    if total > 1_000: return "regular"
    return "casual"

In [ ]:
# `summarize_tiers` CALLS `spending_tier`. Editing the inner helper invalidates
# this cached result even though this cell's own code never changed.
def summarize_tiers(spend):
    return spend.map(spending_tier).value_counts()

tier_counts = summarize_tiers(spend_per_user)
tier_counts

## 6 · Beyond notebooks — the `@cash.cache` decorator

Outside cells, wrap any function with `@cash.cache` and it caches by its
arguments and its own source code. The first call runs; an identical call
returns instantly.

Note the `# @cash:assume-safe` on the `time.sleep` line. cash's analyzer flags
calls that look like side effects — a `sleep` throws its return away, which is
exactly the shape of something that matters and would be skipped on a cache
hit. Here it is deliberate, so the line is waived and the warning goes away.

The waiver is **per line**, on purpose. `@cash.cache(assume_safe=True)` would
silence the whole function including anything added to it next year; annotating
the one line you audited means a new unaudited call still speaks up.

In [ ]:
import time

@cash.cache
def spend_by_category(data, min_amount):
    time.sleep(1.0)  # @cash:assume-safe — the sleep IS the stand-in for slow work
    big = data[data["amount"] >= min_amount]
    return big.groupby("category")["amount"].mean().round(2)

print("first call (runs ~1s):")
print(spend_by_category(transactions, 50))
print("\nsecond call (instant, from cache):")
print(spend_by_category(transactions, 50))

## 7 · It survives a kernel restart

The cache lives on disk, not just in memory. Try **restarting the kernel**, then
**Run all**: the slow pipeline and the simulation restore from cache instead of
recomputing — a fresh kernel picks up right where you left off.

## What did cash save you?

Cache hits, misses, and the wall-clock time cash gave back this session:

In [ ]:
%cash_stats